In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import duckdb
import pandas as pd
import time

DB_PATH = '/content/drive/MyDrive/assignment/ecommerce.duckdb'
con = duckdb.connect(DB_PATH)

def run_query(name, sql):
    """Run query, print execution time and return result df."""
    t0 = time.perf_counter()
    df = con.execute(sql).df()
    elapsed = time.perf_counter() - t0
    print(f"\n{'─'*60}")
    print(f"  {name}")
    print(f"  Execution time: {elapsed:.4f}s")
    print(f"{'─'*60}")
    print(df.head(10).to_string(index=False))
    return df, elapsed

times = {}  # collect all timings for the report

In [ ]:
sql_q1 = """
WITH funnel AS (
    SELECT
        c.category_main,
        COUNT(DISTINCT CASE WHEN fe.event_type = 'view'
              THEN fe.user_id END)                          AS view_users,
        COUNT(DISTINCT CASE WHEN fe.event_type = 'cart'
              THEN fe.user_id END)                          AS cart_users,
        COUNT(DISTINCT CASE WHEN fe.event_type = 'purchase'
              THEN fe.user_id END)                          AS purchase_users
    FROM fact_events fe
    JOIN dim_category c ON fe.category_id = c.category_id
    WHERE c.category_main IS NOT NULL
    GROUP BY c.category_main
)
SELECT
    category_main,
    view_users,
    cart_users,
    purchase_users,
    ROUND(cart_users     * 100.0 / NULLIF(view_users, 0), 2) AS view_to_cart_pct,
    ROUND(purchase_users * 100.0 / NULLIF(cart_users, 0), 2) AS cart_to_purchase_pct
FROM funnel
WHERE view_users > 0
ORDER BY view_users DESC
"""

df_q1, t_q1 = run_query("Q1 — Funnel analysis by category", sql_q1)
times['Q1_funnel'] = t_q1

In [ ]:
sql_q2 = """
SELECT
    fe.user_session,
    fe.user_id,
    COUNT(*)                                                AS total_events,
    COUNT(DISTINCT fe.product_id)                           AS distinct_products,
    ROUND(SUM(CASE WHEN fe.event_type = 'cart'
              THEN fe.price ELSE 0 END), 2)                 AS total_cart_value,
    MAX(CASE WHEN fe.event_type = 'purchase'
        THEN 1 ELSE 0 END)                                  AS purchased
FROM fact_events fe
GROUP BY fe.user_session, fe.user_id
ORDER BY total_events DESC
LIMIT 10
"""

df_q2, t_q2 = run_query("Q2 — Session aggregation", sql_q2)
times['Q2_sessions'] = t_q2

In [ ]:
sql_q3 = """
WITH brand_revenue AS (
    SELECT
        fe.event_month,
        p.brand,
        ROUND(SUM(fe.price), 2)     AS total_revenue,
        COUNT(*)                     AS total_purchases,
        ROW_NUMBER() OVER (
            PARTITION BY fe.event_month
            ORDER BY SUM(fe.price) DESC
        )                            AS rank
    FROM fact_events fe
    JOIN dim_product p ON fe.product_id = p.product_id
    WHERE fe.event_type = 'purchase'
      AND p.brand IS NOT NULL
    GROUP BY fe.event_month, p.brand
)
SELECT
    event_month,
    rank,
    brand,
    total_revenue,
    total_purchases
FROM brand_revenue
WHERE rank <= 10
ORDER BY event_month, rank
"""

df_q3, t_q3 = run_query("Q3 — Top 10 brands by revenue per month", sql_q3)
times['Q3_top_brands'] = t_q3

In [ ]:
sql_q4 = """
WITH oct_buyers AS (
    SELECT DISTINCT user_id
    FROM fact_events
    WHERE event_type  = 'purchase'
      AND event_month = '2019-10'
),
nov_active AS (
    SELECT DISTINCT user_id
    FROM fact_events
    WHERE event_month = '2019-11'
)
SELECT
    o.user_id,
    COUNT(fe.event_id)           AS oct_purchases,
    ROUND(SUM(fe.price), 2)      AS oct_spend
FROM oct_buyers o
LEFT JOIN nov_active n  ON o.user_id = n.user_id
JOIN fact_events fe     ON o.user_id = fe.user_id
                       AND fe.event_type  = 'purchase'
                       AND fe.event_month = '2019-10'
WHERE n.user_id IS NULL          -- never appeared in November
GROUP BY o.user_id
ORDER BY oct_spend DESC
LIMIT 10
"""

df_q4, t_q4 = run_query("Q4 — Oct purchasers who did not return in Nov", sql_q4)
times['Q4_churn'] = t_q4

# also print total churned user count
total_churned = con.execute("""
    SELECT COUNT(*) FROM (
        SELECT DISTINCT user_id FROM fact_events
        WHERE event_type = 'purchase' AND event_month = '2019-10'
    ) oct
    WHERE oct.user_id NOT IN (
        SELECT DISTINCT user_id FROM fact_events WHERE event_month = '2019-11'
    )
""").fetchone()[0]
print(f"\nTotal churned users (Oct buyers not in Nov): {total_churned:,}")

In [ ]:
sql_q5 = """
SELECT
    EXTRACT(HOUR FROM event_time)::INT      AS hour_of_day,
    COUNT(*)                                AS total_purchases,
    ROUND(COUNT(*) * 100.0 /
          SUM(COUNT(*)) OVER (), 2)         AS pct_of_daily
FROM fact_events
WHERE event_type = 'purchase'
GROUP BY hour_of_day
ORDER BY hour_of_day
"""

df_q5, t_q5 = run_query("Q5 — Hourly purchase distribution", sql_q5)
times['Q5_hourly'] = t_q5

peak_hour = df_q5.loc[df_q5['total_purchases'].idxmax()]
print(f"\nPeak purchase hour: {int(peak_hour['hour_of_day']):02d}:00  "
      f"({int(peak_hour['total_purchases']):,} purchases, "
      f"{peak_hour['pct_of_daily']}% of daily)")

In [ ]:
summary = pd.DataFrame([
    {'Query': k, 'Time with index (s)': round(v, 4)}
    for k, v in times.items()
])

# pull no-index times from benchmark results saved earlier
# (if you ran Cell 6 of 03_benchmarks, these are in idx_df)
# otherwise rerun 3 queries without indexes here for completeness

print("\nQuery execution time summary")
print(summary.to_string(index=False))
print(f"\nTotal query time: {sum(times.values()):.4f}s")

In [ ]:
import os

OUT = '/content/drive/MyDrive/assignment/reports/'
os.makedirs(OUT, exist_ok=True)

df_q1.to_csv(f'{OUT}q1_funnel.csv',       index=False)
df_q2.to_csv(f'{OUT}q2_sessions.csv',     index=False)
df_q3.to_csv(f'{OUT}q3_top_brands.csv',   index=False)
df_q4.to_csv(f'{OUT}q4_churn.csv',        index=False)
df_q5.to_csv(f'{OUT}q5_hourly.csv',       index=False)
summary.to_csv(f'{OUT}query_timings.csv', index=False)

print("All query results saved ✓")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(df_q5['hour_of_day'], df_q5['total_purchases'],
       color='#7F77DD', width=0.7)
ax.axvline(x=peak_hour['hour_of_day'], color='#D85A30',
           linestyle='--', linewidth=1.5, label=f"Peak: {int(peak_hour['hour_of_day']):02d}:00")
ax.set_title('Hourly purchase distribution (Oct + Nov combined)', fontsize=13)
ax.set_xlabel('Hour of day (UTC)')
ax.set_ylabel('Total purchases')
ax.set_xticks(range(0, 24))
ax.set_xticklabels([f'{h:02d}:00' for h in range(24)], rotation=45, ha='right', fontsize=8)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT}q5_hourly_chart.png', dpi=150)
plt.show()
print("Hourly chart saved ✓")